# SciSum-Qwen Colab Runner

This notebook mounts Google Drive, unpacks the project zip, installs dependencies, runs a real Qwen baseline on a subset, evaluates it, and optionally launches a small QLoRA training run.

## 1. Mount Google Drive

Upload `dist/scisum-qwen-colab-ready.zip` to your Drive first. A convenient location is `MyDrive/scisum-qwen-colab/dist/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/scisum-qwen-colab'
PROJECT_ZIP = f'{DRIVE_ROOT}/dist/scisum-qwen-colab-ready.zip'
WORKSPACE = '/content/scisum-qwen'

assert os.path.exists(PROJECT_ZIP), f'Missing zip: {PROJECT_ZIP}'

In [ ]:
!rm -rf "$WORKSPACE"
!mkdir -p "$WORKSPACE"
!unzip -q "$PROJECT_ZIP" -d /content
%cd /content/scisum-qwen

## 2. Install Dependencies

Use a GPU runtime in Colab before running this notebook.

In [ ]:
!python -V
!nvidia-smi || true
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation -e .

## 3. Quick Verification

In [ ]:
!python -m pytest tests/test_api.py tests/test_service.py tests/test_demo.py

## 4. Real Qwen Baseline on the Colab Subset

In [ ]:
!PYTHONPATH=src python -m scisum_qwen.evaluation.run_baselines \
  --input data/colab/test_subset.jsonl \
  --output-dir reports/colab_runs/baseline \
  --model-name Qwen/Qwen2.5-3B-Instruct \
  --limit 10 \
  --max-source-words 1200 \
  --max-new-tokens 160

In [ ]:
!PYTHONPATH=src python -m scisum_qwen.evaluation.run_eval \
  --input-dir reports/colab_runs/baseline \
  --output-csv reports/colab_runs/baseline_metrics.csv \
  --output-md reports/colab_runs/baseline_eval.md \
  --manual-review-csv reports/colab_runs/manual_review_template.csv

## 5. Optional: Small Real QLoRA Run

In [ ]:
!PYTHONPATH=src python -m scisum_qwen.training.train_qlora \
  --config configs/train_qlora_colab.yaml

## 6. Save Artifacts Back to Drive

In [ ]:
!mkdir -p "$DRIVE_ROOT/outputs"
!cp -r reports/colab_runs "$DRIVE_ROOT/outputs/"
!cp -r models "$DRIVE_ROOT/outputs/" || true
!cp reports/colab_training_log.md "$DRIVE_ROOT/outputs/" || true